In [1]:
import pandas as pd 
import os

In [19]:
import os
import pandas as pd
import numpy as np

root_dir = "../generated_data/csv"
X_list = []  # Stores time-series data in sktime format
y_list = []  # Stores labels

n_timestamps = 2500 # Fixed number of timestamps

# Iterate through each subdirectory (which represents the type of failure)
for subdir in os.listdir(root_dir):
    subdir_path = os.path.join(root_dir, subdir)

    if os.path.isdir(subdir_path):  # Ensure it's a directory
        for file in sorted(os.listdir(subdir_path)):
            if file.endswith(".csv"):
                file_path = os.path.join(subdir_path, file)

                # Read CSV
                df = pd.read_csv(file_path)

                # Ensure all columns are numerical
                df = df.apply(pd.to_numeric, errors='coerce').fillna(0)

                # Ensure all time series have a fixed length
                if len(df) < n_timestamps:
                    # Pad with zeros if fewer timestamps
                    pad_data = pd.DataFrame([[0] * len(df.columns)] * (n_timestamps - len(df)), columns=df.columns)
                    df = pd.concat([df, pad_data], ignore_index=True)
                elif len(df) > n_timestamps:
                    # Truncate to the fixed number of timestamps
                    df = df.head(n_timestamps)

                # Convert to sktime nested DataFrame format (each column contains a pandas Series)
                nested_df = pd.DataFrame({col: [pd.Series(df[col].values)] for col in df.columns})

                # Append the formatted data
                X_list.append(nested_df)
                y_list.append(subdir)  # Class label

# Combine all instances into a single DataFrame
X = pd.concat(X_list, ignore_index=True)
y = pd.Series(y_list)

# df = X
# df["type"] = y

# df.to_csv("new-dataset.csv")


# Now X and y are ready for use with DRCIF
print(X.head())
print(y.head())


                                             Time(s)  \
0  0       0.0000
1       0.0001
2       0.0002
3...   
1  0       0.0000
1       0.0001
2       0.0002
3...   
2  0       0.0000
1       0.0001
2       0.0002
3...   
3  0       0.0000
1       0.0001
2       0.0002
3...   
4  0       0.0000
1       0.0001
2       0.0002
3...   

                                                  Kn  \
0  0       141.57462
1       141.57503
2       14...   
1  0       276.34406
1       276.34553
2       27...   
2  0       104.24892
1       104.24969
2       10...   
3  0       420.31945
1       420.32630
2       40...   
4  0       135.82012
1       135.82341
2       13...   

                                Chamber Pressure(Pa)  \
0  0       1.271278e+06
1       1.271284e+06
2   ...   
1  0       3.567544e+06
1       3.567571e+06
2   ...   
2  0       1.185893e+06
1       1.185907e+06
2   ...   
3  0       1.247696e+07
1       1.247726e+07
2   ...   
4  0       2.284586e+06
1       2.284675e+06
2

In [20]:
from sktime.classification.interval_based import TimeSeriesForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


from sktime.classification.interval_based import DrCIF

dr = DrCIF()
dr.fit(X_train, y_train)

/home/rihaan/miniconda3/lib/python3.12/site-packages/sktime/transformations/panel/catch22.py:390: FutureWarning: In Catch22._transform_single_feature, the argument case_id is deprecated and will be removed in the future.
  warn(
/home/rihaan/miniconda3/lib/python3.12/site-packages/sktime/transformations/panel/catch22.py:390: FutureWarning: In Catch22._transform_single_feature, the argument case_id is deprecated and will be removed in the future.
  warn(
/home/rihaan/miniconda3/lib/python3.12/site-packages/sktime/transformations/panel/catch22.py:390: FutureWarning: In Catch22._transform_single_feature, the argument case_id is deprecated and will be removed in the future.
  warn(
/home/rihaan/miniconda3/lib/python3.12/site-packages/sktime/transformations/panel/catch22.py:390: FutureWarning: In Catch22._transform_single_feature, the argument case_id is deprecated and will be removed in the future.
  warn(
/home/rihaan/miniconda3/lib/python3.12/site-packages/sktime/transformations/panel/ca

DrCIF()

In [21]:
y_pred = dr.predict(X_test)

/home/rihaan/miniconda3/lib/python3.12/site-packages/sktime/transformations/panel/catch22.py:390: FutureWarning: In Catch22._transform_single_feature, the argument case_id is deprecated and will be removed in the future.
  warn(


In [22]:
from sklearn.metrics import accuracy_score, f1_score

In [23]:
ac = accuracy_score(y_test, y_pred)

In [24]:
print(ac)

0.3333333333333333
